In [1]:
import os
import glob
import pandas as pd
import easyocr
import re
from pathlib import Path

In [2]:
BASE_DIR = "/home/hasan/coding/MoneyLens/ai/Dataset_ocr/preprocessed"
SPLITS   = ["train", "valid", "test"]

# init OCR (CPU biar aman)
reader = easyocr.Reader(['en'], gpu=False)

def clean_text(text):
    text = text.strip()
    text = re.sub(r'[^0-9a-zA-Z.,:/\- ]', '', text)  # buang karakter aneh
    return text

results = []

print("="*60)
print("GENERATE GROUND TRUTH DARI CROPS (EasyOCR)")
print("="*60)

for split in SPLITS:
    crops_dir = os.path.join(BASE_DIR, split, "crops")

    if not os.path.exists(crops_dir):
        print(f"[{split}] folder tidak ada")
        continue

    img_paths = glob.glob(os.path.join(crops_dir, "*.png"))

    if not img_paths:
        print(f"[{split}] tidak ada gambar")
        continue

    print(f"\n[{split}] {len(img_paths)} gambar")

    for img_path in img_paths:
        fname = Path(img_path).name

        try:
            # OCR (lebih stabil)
            ocr_result = reader.readtext(img_path, detail=0, paragraph=False)

            # gabung teks
            text = " ".join(ocr_result).strip()

            # cleaning
            text = clean_text(text)

            if "total_transaksi" in fname:
                text = re.sub(r'[^0-9.]', '', text)
            
            elif "tanggal" in fname:
                text = re.sub(r'[^0-9/-]', '', text)
            
            # filter hasil jelek
            if text == "" or len(text) < 2:
                text = ""

            results.append({
                "split": split,
                "filename": fname,
                "text": text
            })

        except Exception as e:
            print(f"[ERROR] {fname}: {e}")

            results.append({
                "split": split,
                "filename": fname,
                "text": ""
            })

Using CPU. Note: This module is much faster with a GPU.


GENERATE GROUND TRUTH DARI CROPS (EasyOCR)

[train] 3194 gambar

[valid] 915 gambar

[test] 422 gambar


In [3]:
df = pd.DataFrame(results)

out_path = os.path.join(BASE_DIR, "ground_truth_auto.csv")
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("\n" + "="*60)
print(f"SELESAI → {out_path}")
print("="*60)


SELESAI → /home/hasan/coding/MoneyLens/ai/Dataset_ocr/preprocessed/ground_truth_auto.csv
